# Two-input dataset

Building a multi-input model starts with crafting a custom dataset that can supply all the inputs to the model. In this exercise, you will build the Omniglot dataset that serves triplets consisting of:

- The image of a character to be classified,
- The one-hot encoded alphabet vector of length 30, with zeros everywhere but for a single one denoting the ID of the alphabet the character comes from,
- The target label, an integer between 0 and 963.

In [1]:
import pandas as pd
# samples = pd.read_csv("dataset/omniglot_train.csv") # Should be converted to list
# contains data like:
#[('/usr/local/share/datasets/omniglot_train/Gujarati/character42/0459_16.png',
#  array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
#         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32),
#  0),
# ('/usr/local/share/datasets/omniglot_train/Bengali/character42/0459_02.png',
#  array([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
#         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32),
#  0), ....

In [2]:
# from PIL import Image
# from torch.utils.data import DataLoader, Dataset
# from torchvision import transforms

# class OmniglotDataset(Dataset):
#     def __init__(self, transform, samples):
# 		# Assign transform and samples to class attributes
#         self.transform = transform
#         self.samples = samples
                    
#     def __len__(self):
# 		# Return number of samples
#         return len(self.samples)

#     def __getitem__(self, idx):
#       	# Unpack the sample at index idx
#         img_path, alphabet, label = self.samples[idx]
#         img = Image.open(img_path).convert('L')
#         # Transform the image 
#         img_transformed = self.transform(img)
#         return img_transformed, alphabet, label

# dataset_train = OmniglotDataset(
#     transform=transforms.Compose([
#         transforms.ToTensor(),
#         transforms.Resize((64, 64)),
#     ]),
#     samples=samples,
# )

# dataloader_train = DataLoader(
#     dataset_train, shuffle=True, batch_size=3,
# )

# Two-input model

With the data ready, it's time to build the two-input model architecture! To do so, you will set up a model class with the following methods:
- `.__init__()`, in which you will define sub-networks by grouping layers; this is where you define the two layers for processing the two inputs, and the classifier that returns a classification score for each class.
- `forward()`, in which you will pass both inputs through corresponding pre-defined sub-networks, concatenate the outputs, and pass them to the classifier.

In [4]:
import torch
from torch import nn

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Define sub-networks as sequential models
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(16*32*32, 128)
        )
        self.alphabet_layer = nn.Sequential(
            nn.Linear(30, 8),
            nn.ELU(), 
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 + 8, 964), 
        )
        
    def forward(self, x_image, x_alphabet):
		# Pass the x_image and x_alphabet through appropriate layers
        x_image = self.image_layer(x_image)
        x_alphabet = self.alphabet_layer(x_alphabet)
        # Concatenate x_image and x_alphabet
        x = torch.cat((x_image, x_alphabet), dim=1)
        return self.classifier(x)

# Training two-input model

he training loop for your two-input model will be a typical PyTorch training loop. The only change compared to the training loops you have written before is that now you have two inputs instead of one.

What is the correct way of iterating through the `dataloader` and passing the inputs to the model?

- `for img, alpha, labels in dataloader_train: outputs = net(img, alpha)`

# Two-output Dataset and DataLoader

In this and the following exercises, you will build a two-output model to predict both the character and the alphabet it comes from based on the character's image. As always, you will start with getting the data ready.

In [5]:
# from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms

# # Print the sample at index 100
# print(samples[100])

# # Print the sample at index 100
# print(samples[100])

# # Create dataset_train
# dataset_train = OmniglotDataset( # OmniglotDataset is derived class of Dataset / inherits Dataset class of pytorch utils.data
#     transform=transforms.Compose([
#         transforms.ToTensor(),
#       	transforms.Resize((64, 64)),
#     ]),
#     samples=samples,
# )

# # Create dataloader_train
# dataloader_train = DataLoader(
#     dataset_train, batch_size=32, shuffle=True,
# )

# Two-output model architecture

In this exercise, you will construct a multi-output neural network architecture capable of predicting the character and the alphabet.

Recall the general structure: in the `.__init__()` method, you define layers to be used in the forward pass later. In the `forward()` method, you will first pass the input image through a couple of layers to obtain its embedding, which in turn is fed into two separate classifier layers, one for each output.

In [6]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(16*32*32, 128)
        )
        # Define the two classifier layers
        self.classifier_alpha = nn.Linear(128, 30)
        self.classifier_char = nn.Linear(128, 964)
        
    def forward(self, x):
        x_image = self.image_layer(x)
        # Pass x_image through the classifiers and return both results
        output_alpha = self.classifier_alpha(x_image)
        output_char = self.classifier_char(x_image)
        return output_alpha, output_char

# Training multi-output models

When training models with multiple outputs, it is crucial to ensure that the loss function is defined correctly.

In this case, the model produces two outputs: predictions for the alphabet and the character. For each of these, there are corresponding ground truth labels, which will allow you to calculate two separate losses: one incurred from incorrect alphabet classifications, and the other from incorrect character classification. Since in both cases you are dealing with a multi-label classification task, the Cross-Entropy loss can be applied each time.

Gradient descent can optimize only one loss function, however. You will thus define the total loss as the sum of alphabet and character losses.

In [8]:
# from torch import optim 
# net = Net()
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(net.parameters(), lr=0.05)

# for epoch in range(1):
#     for images, labels_alpha, labels_char in dataloader_train:
#         optimizer.zero_grad()
#         outputs_alpha, outputs_char = net(images)
#         # Compute alphabet classification loss
#         loss_alpha = criterion(outputs_alpha, labels_alpha)
#         # Compute character classification loss
#         loss_char = criterion(outputs_char, labels_char)
#         # Compute total loss
#         loss = loss_alpha + loss_char
#         loss.backward()
#         optimizer.step()

# Multi-output model evaluation

In this exercise, you will practice model evaluation for multi-output models. Your task is to write a function called evaluate_model() that takes an alphabet-and-character-predicting model as input, runs the evaluation loop, and prints the model's accuracy in the two tasks.

In [9]:
# import torch
# from torchmetrics import Accuracy

# def evaluate_model(model):
#     # Define accuracy metrics
#     acc_alpha = Accuracy(task="multiclass", num_classes=30)
#     acc_char = Accuracy(task="multiclass", num_classes=964)

#     model.eval()
#     with torch.no_grad():
#         for images, labels_alpha, labels_char in dataloader_test:
#             # Obtain model outputs
#             outputs_alpha, outputs_char = model(images)
#             _, pred_alpha = torch.max(outputs_alpha, 1)
#             _, pred_char = torch.max(outputs_char, 1)
# 			# Update both accuracy metrics
#             acc_alpha(pred_alpha, labels_alpha)
#             acc_char(pred_char, labels_char)
    
#     print(f"Alphabet: {acc_alpha.compute()}")
#     print(f"Character: {acc_char.compute()}")

# Loss weighting

Three versions of the two-output model for alphabet and character prediction that you built before have been trained: model_a, model_b, and model_c. For all three, the loss was defined as follows:
```
loss_alpha = criterion(outputs_alpha, labels_alpha)
loss_char = criterion(outputs_char, labels_char)
loss = ((1 - char_weight) * loss_alpha) + (char_weight * loss_char)
```

However, each of the three models was trained with a different char_weight: 0.1, 0.5, or 0.9.

In [10]:
# evaluate_model(model_a)
# evaluate_model(model_b)
# evaluate_model(model_c)

- `model_a`: 0.5, `model_b`: 0.1, `model_c`: 0.9
